In [1]:
import sqlite3
import pandas as pd

In [2]:
CONN = sqlite3.connect("E://irs990_full.db")

# ! Remove the limit on `filing_base_q` to run full query

In [ ]:
filings_base_q = """
    select *
    from (
        select
            *,
            row_number() over(PARTITION by ein, tax_year order by tax_period_end_date desc, return_timestamp desc, form_type_rank asc) as rank
        from (
            select
                tax_year,
                tax_period_end_date,
                return_timestamp,
                filing_id,
                ein,
                case
                    when form_type = '990' then 1
                    when form_type = '990EZ' then 2
                    when form_type = '990PF' then 3
                    when form_type = '990T' then 4
                    else 5
                end as form_type_rank
            from irs990_filings
        ) f
    ) f2
    where f2.rank = 1
"""

features_base_q = """
select
	f.filing_id, f.ein, f.tax_year, upper(f.filer_name) as current_filer_name,
    form_type, exempt_organization_type,
	case
		when form_type = '990' then 1
		else 0
	end as has_990,
	case
		when form_type = '990EZ' then 1
		else 0
	end as has_990EZ,
	case
		when form_type = '990PF' then 1
		else 0
	end as has_990PF,
	case
		when form_type = '990T' then 1
		else 0
	end as has_990T,
	mission,
	case
		when mission is null then 0
		else 1
	end as has_mission,
	null as mission_words,
	case
		when total_revenue is not null and total_revenue <> 0 then grants_and_contributions / total_revenue
		else 0
	end as grants_as_rev_frac,
	case
		when voting_members_governing_body is not null and voting_members_governing_body <> 0 then voting_members_independent / voting_members_governing_body
		else null
	end as perc_ind_voting_members,
    political_activity_flag,
    num_contractors,
    c.total_compensation as total_contractor_comp,
    case
        when total_expenses is not null and total_expenses > 0 then c.total_compensation / total_expenses
        else null
    end as contractor_compensation_frac,
    c.avg_compensation as mean_contractor_comp,
    total_expenses,
    num_grants,
    g.total_granted, g.avg_granted,
    l.total_lobbying_expenditures_amt, l.fees_for_services_lobbying_amt,
    coalesce(filer_address_line1, '') || ' ' || coalesce(filer_address_line2, '') || ' ' || coalesce(filer_city, '') || ' ' || coalesce(filer_state, '') || ' ' || coalesce(filer_zip, filer_zip_code, '') as address,
    doing_business_as_name
from irs990_filings f
	left join (
		select filing_id, count(line_no) as num_contractors, sum(compensation) as total_compensation, avg(compensation) as avg_compensation
		from irs990_filing_contractors
		where lower(trim(contractor_name)) not in ('na', 'none', 'n/a', 'n a')
		group by filing_id
	) c on f.filing_id = c.filing_id
	left join (
		select filing_id, count(line_no) as num_grants, sum(amount) as total_granted, avg(amount) as avg_granted
		from irs990_filing_grants
		group by filing_id
	) g on f.filing_id = g.filing_id
	left join irs990_filing_lobbying l on f.filing_id = l.filing_id
"""

out_types = [
    'B', # Grant to related org
    'D', # Loan to related org
    'G', # Sale of assets to related org
    'J', # Lease of facilities, etc. to related org
    'L', # Performance of services, etc. for related org
    'P', # Reimbursement paid to related org
    'R'  # Other transfer of cash etc. to related org
]

in_types = [
    'C', # Grant from related org
    'E', # Loan from related org
    'H', # Purchase of assets from related org
    'K', # Lease of facilities, etc. from related org
    'M', # Performance of services, etc. from related org
    'Q', # Reimbursement paid from related org
    'S'  # Other transfer of cash etc. from related org
]

between_types = [
    'I', # Exhange of assets between related orgs
    'N', # Sharing of facilities, etc. between related orgs
    'O'  # Sharing of paid employees w. related orgs
]
org_transactions_base_q = f"""
select *
from (
	select 
		filing_id, related_org_name,
		case
			when type in ('{"','".join(out_types)}') then count(line_no)
			else null
		end as num_out_transactions,
		case
			when type in ('{"','".join(out_types)}') then sum(amount)
			else null
		end as sum_out_transactions,
		case
			when type in ('{"','".join(out_types)}') then avg(amount)
			else null
		end as avg_out_transactions,
		--In
		case
			when type in ('{"','".join(in_types)}') then count(line_no)
			else null
		end as num_in_transactions,
		case
			when type in ('{"','".join(in_types)}') then sum(amount)
			else null
		end as sum_in_transactions,
		case
			when type in ('{"','".join(in_types)}') then avg(amount)
			else null
		end as avg_in_transactions,
		-- Between
		case
			when type in ('{"','".join(between_types)}') then count(line_no)
			else null
		end as num_between_transactions,
		case
			when type in ('{"','".join(between_types)}') then sum(amount)
			else null
		end as sum_between_transactions,
		case
			when type in ('{"','".join(between_types)}') then avg(amount)
			else null
		end as avg_between_transactions
	from irs990_filing_related_org_transactions
	where
		type not in (
			'A', -- Receipts from controlled entity
			'F' -- Dividends from related org
		)
	group by filing_id, related_org_name
) org_transactions

"""
org_people_q = """
select 
	filing_id,
	sum(is_indiv_trustee_or_director) as num_indiv_trustee_or_director,
	sum(is_institutional_trustee) as num_institutional_trustees,
	sum(is_officer) as num_officers,
	sum(is_key_employee) as num_key_employees,
	sum(is_former_employee) as num_former_employees,
	--From Org
	min(avg_weekly_hours_worked_org) as min_avg_weekly_hours_org,
	avg(avg_weekly_hours_worked_org) as avg_avg_weekly_hours_org,
	max(avg_weekly_hours_worked_org) as max_avg_weekly_hours_org,
	min(compensation_from_org) as min_compensation_org,
	avg(compensation_from_org) as avg_compensation_org,
	max(compensation_from_org) as max_compensation_org,
	--From Related Org
	min(avg_weekly_hours_worked_related_org) as min_avg_weekly_hours_related_org,
	avg(avg_weekly_hours_worked_related_org) as avg_avg_weekly_hours_related_org,
	max(avg_weekly_hours_worked_related_org) as max_avg_weekly_hours_related_org,
	min(compensation_from_related_org) as min_compensation_related_org,
	avg(compensation_from_related_org) as avg_compensation_related_org,
	max(compensation_from_related_org) as max_compensation_related_org,
	--From Other
	min(compensation_other) as min_compensation_other,
	avg(compensation_other) as avg_compensation_other,
	max(compensation_other) as max_compensation_other
from irs990_filing_people
group by filing_id

"""



In [125]:
features_base_df = pd.read_sql(features_base_q, CONN)
features_base_df

,filing_id,ein,tax_year,current_filer_name,form_type,exempt_organization_type,has_990,has_990EZ,has_990PF,has_990T,...,contractor_compensation_frac,mean_contractor_comp,total_expenses,num_grants,total_granted,avg_granted,total_lobbying_expenditures_amt,fees_for_services_lobbying_amt,address,doing_business_as_name
0,1,272292010,2019,NORTH CENTRAL ACADEMY,990,501(c)(3),1,0,0,0,...,0.869091,988913.0,1137870.0,NaN,NaN,NaN,NaN,NaN,928 W MARKET STREET TIFFIN OH 44883,NaN
1,2,911152733,2020,SHEIKH ABDUL KADIR IDRESS MOSQUE TRUST,990,501(c)(2),1,0,0,0,...,NaN,NaN,5794.0,NaN,NaN,NaN,NaN,NaN,1420 NE NORTHGATE WAY SEATTLE WA 98125,NaN
2,3,223519265,2020,FRIENDS OF HOBOKEN CHARTER SCHOOL INC,990,501(c)(3),1,0,0,0,...,NaN,NaN,25534.0,NaN,NaN,NaN,NaN,NaN,713 WASHINGTON STREET HOBOKEN NJ 07030,NaN
3,4,260741074,2020,NORTH CAROLINA TAX COLLECTORS ASSOCIATION,990EZ,501(c)(3),0,1,0,0,...,NaN,NaN,28827.0,NaN,NaN,NaN,NaN,NaN,65 Glen Road Box 246 Garner NC 27529,NaN
4,5,814522819,2020,CHUA PHAP NGHIEM INC,990EZ,501(c)(3),0,1,0,0,...,NaN,NaN,85049.0,NaN,NaN,NaN,NaN,NaN,1610 TODDS LN Hampton VA 23666,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,96,431879103,2020,MISSOURI ASSOCIATION OF COUNTY CLERKS AND ELEC...,990EZ,501(c)(6),0,1,0,0,...,NaN,NaN,107641.0,NaN,NaN,NaN,NaN,NaN,PO BOX 248 MAYSVILLE MO 64469,NaN
96,97,320078573,2020,ARMENIAN INTERNATIONAL MEDICAL FUND,990,501(c)(3),1,0,0,0,...,NaN,NaN,301122.0,NaN,NaN,NaN,NaN,NaN,14960 DICKENS STREET 301 SHERMAN OAKS CA 91403,NaN
97,98,760190638,2019,OPTIMIST CLUB FOUNDATION OF BAYTOWN,990EZ,501(c)(3),0,1,0,0,...,NaN,NaN,285.0,NaN,NaN,NaN,NaN,NaN,P O BOX 695 BAYTOWN TX 77522,NaN
98,99,223681722,2020,RALLYCAP SPORTS INC,990,501(c)(3),1,0,0,0,...,NaN,NaN,184088.0,NaN,NaN,NaN,NaN,NaN,400 BEACON BLVD SEA GIRT NJ 08750,NaN


In [126]:
form_type = features_base_df.groupby(['ein', 'tax_year'], as_index=False).agg(
    has_990=('has_990', 'max'),
    has_990EZ=('has_990EZ', 'max'),
    has_990PF=('has_990PF', 'max'),
    has_990T=('has_990T', 'max')
)
display(form_type)

mission = features_base_df.loc[:, ['ein', 'tax_year', 'has_mission', 'mission', 'mission_words']]
mission['mission_words'] = mission.mission.str.split().str.len()

mission = mission.groupby(['ein', 'tax_year'], as_index=False).agg(
    has_mission=('has_mission', 'max'),
    mission_words=('mission_words', 'max'),
)
display(mission)

address = features_base_df.loc[:, ['ein', 'tax_year', 'address']]
address = address.groupby('ein', as_index=False).agg(
    min_year=('tax_year', 'min'),
    max_year=('tax_year', 'max'),
    unique_addresses=('address', 'nunique')
)
address['lifetime_avg_address_changes'] = address.apply(lambda row: max(0, row.unique_addresses - 1) / (row.max_year - row.min_year + 1), axis=1)
address = address.drop(['min_year', 'max_year'], axis=1).drop_duplicates()
display(address)

name = features_base_df.loc[:, ['ein', 'tax_year', 'doing_business_as_name', 'current_filer_name']]
name = name.groupby('ein', as_index=False).agg(
    min_year=('tax_year', 'min'),
    max_year=('tax_year', 'max'),
    unique_dbas=('doing_business_as_name', 'nunique'),
    unique_names=('current_filer_name', 'nunique')
)
name['lifetime_avg_unique_dbas'] = name.apply(lambda row: max(0, row.unique_dbas - 1) / (row.max_year - row.min_year + 1), axis=1)
name['lifetime_avg_unique_names'] = name.apply(lambda row: max(0, row.unique_names - 1) / (row.max_year - row.min_year + 1), axis=1)
name = name.drop(['min_year', 'max_year'], axis=1).drop_duplicates()
name

,ein,tax_year,has_990,has_990EZ,has_990PF,has_990T
0,010884807,2020,0,0,1,0
1,020649872,2020,0,0,1,0
2,020672951,2020,1,0,0,0
3,042429311,2020,1,0,0,0
4,043204112,2020,1,0,0,0
...,...,...,...,...,...,...
95,880132649,2020,1,0,0,0
96,910839740,2019,0,1,0,0
97,911098072,2020,0,1,0,0
98,911152733,2020,1,0,0,0


,ein,tax_year,has_mission,mission_words
0,010884807,2020,0,NaN
1,020649872,2020,0,NaN
2,020672951,2020,1,9.0
3,042429311,2020,1,29.0
4,043204112,2020,1,59.0
...,...,...,...,...
95,880132649,2020,1,94.0
96,910839740,2019,1,18.0
97,911098072,2020,1,5.0
98,911152733,2020,1,10.0


,ein,unique_addresses,lifetime_avg_address_changes
0,010884807,1,0.0
1,020649872,1,0.0
2,020672951,1,0.0
3,042429311,1,0.0
4,043204112,1,0.0
...,...,...,...
95,880132649,1,0.0
96,910839740,1,0.0
97,911098072,1,0.0
98,911152733,1,0.0


,ein,unique_dbas,unique_names,lifetime_avg_unique_dbas,lifetime_avg_unique_names
0,010884807,0,1,0.0,0.0
1,020649872,0,1,0.0,0.0
2,020672951,0,1,0.0,0.0
3,042429311,0,1,0.0,0.0
4,043204112,0,1,0.0,0.0
...,...,...,...,...,...
95,880132649,0,1,0.0,0.0
96,910839740,0,1,0.0,0.0
97,911098072,0,1,0.0,0.0
98,911152733,0,1,0.0,0.0


In [127]:
unique_filings = pd.read_sql(filings_base_q, CONN)

In [128]:
features = features_base_df.copy()
features = features.drop(['current_filer_name', 'form_type', 'has_990', 'has_990EZ', 'has_990PF', 'has_990T', 'mission', 'has_mission', 'mission_words', 'address', 'doing_business_as_name', 'current_filer_name'], axis=1)
features.merge(
    unique_filings.loc[:, ['filing_id']],
    on='filing_id',
    how='inner'
)

,filing_id,ein,tax_year,exempt_organization_type,grants_as_rev_frac,perc_ind_voting_members,political_activity_flag,num_contractors,total_contractor_comp,contractor_compensation_frac,mean_contractor_comp,total_expenses,num_grants,total_granted,avg_granted,total_lobbying_expenditures_amt,fees_for_services_lobbying_amt
0,1,272292010,2019,501(c)(3),0.673025,1.0,0.0,1.0,988913.0,0.869091,988913.0,1137870.0,NaN,NaN,NaN,NaN,NaN
1,2,911152733,2020,501(c)(2),0.363716,0.0,0.0,NaN,NaN,NaN,NaN,5794.0,NaN,NaN,NaN,NaN,NaN
2,3,223519265,2020,501(c)(3),0.072674,1.0,0.0,NaN,NaN,NaN,NaN,25534.0,NaN,NaN,NaN,NaN,NaN
3,4,260741074,2020,501(c)(3),NaN,NaN,0.0,NaN,NaN,NaN,NaN,28827.0,NaN,NaN,NaN,NaN,NaN
4,5,814522819,2020,501(c)(3),NaN,NaN,0.0,NaN,NaN,NaN,NaN,85049.0,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
94,96,431879103,2020,501(c)(6),NaN,NaN,0.0,NaN,NaN,NaN,NaN,107641.0,NaN,NaN,NaN,NaN,NaN
95,97,320078573,2020,501(c)(3),0.993314,0.0,0.0,NaN,NaN,NaN,NaN,301122.0,NaN,NaN,NaN,NaN,NaN
96,98,760190638,2019,501(c)(3),NaN,NaN,0.0,NaN,NaN,NaN,NaN,285.0,NaN,NaN,NaN,NaN,NaN
97,99,223681722,2020,501(c)(3),0.914130,1.0,0.0,NaN,NaN,NaN,NaN,184088.0,NaN,NaN,NaN,NaN,NaN


In [129]:
ot = pd.read_sql(org_transactions_base_q, CONN).fillna(0)
ot['passthrough'] = (ot.sum_out_transactions + ot.sum_between_transactions) / ((ot.sum_in_transactions + ot.sum_between_transactions)+1)
ot['transactions'] = ot.num_out_transactions + ot.sum_in_transactions + ot.sum_between_transactions
ot = ot.groupby('filing_id', as_index=False).agg(
    min_passthrough=('passthrough', 'min'),
    avg_passthrough=('passthrough', 'mean'),
    max_passthrough=('passthrough', 'max'),
    total_passthrough=('passthrough', 'sum'),
    min_trxns=('transactions', 'min'),
    avg_trxns=('transactions', 'mean'),
    max_trxns=('transactions', 'max'),
    total_trxns=('transactions', 'sum')
)
ot

,filing_id,min_passthrough,avg_passthrough,max_passthrough,total_passthrough,min_trxns,avg_trxns,max_trxns,total_trxns
0,9,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.0,0.000000e+00,0.0,0.0
1,21,0.000000e+00,3.094743e+07,3.386638e+08,3.404217e+08,1.0,9.523178e+05,5231061.0,10475496.0
2,33,9.861540e+05,8.545041e+06,1.727405e+07,2.563512e+07,1.0,1.000000e+00,1.0,3.0
3,39,1.325301e+08,1.325301e+08,1.325301e+08,1.325301e+08,5.0,5.000000e+00,5.0,5.0
4,45,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,9153726.0,9.153726e+06,9153726.0,9153726.0
...,...,...,...,...,...,...,...,...,...
503,8651,3.475810e+05,3.475810e+05,3.475810e+05,3.475810e+05,1.0,1.000000e+00,1.0,1.0
504,8696,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,98266.0,9.826600e+04,98266.0,98266.0
505,8699,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,1.0,1.000000e+00,1.0,1.0
506,8712,9.999770e-01,9.999770e-01,9.999770e-01,9.999770e-01,43519.0,4.351900e+04,43519.0,43519.0


In [130]:
people = pd.read_sql(org_people_q, CONN)
people

,filing_id,num_indiv_trustee_or_director,num_institutional_trustees,num_officers,num_key_employees,num_former_employees,min_avg_weekly_hours_org,avg_avg_weekly_hours_org,max_avg_weekly_hours_org,min_compensation_org,...,max_compensation_org,min_avg_weekly_hours_related_org,avg_avg_weekly_hours_related_org,max_avg_weekly_hours_related_org,min_compensation_related_org,avg_compensation_related_org,max_compensation_related_org,min_compensation_other,avg_compensation_other,max_compensation_other
0,1,5.0,NaN,5.0,NaN,None,2.00,8.750000,40.00,0.0,...,0.0,NaN,NaN,NaN,400.0,16920.625,99665.0,0.0,0.000000,0.0
1,2,1.0,NaN,NaN,NaN,None,5.00,5.000000,5.00,0.0,...,0.0,NaN,NaN,NaN,0.0,0.000,0.0,0.0,0.000000,0.0
2,3,5.0,NaN,6.0,NaN,None,1.00,1.555556,2.00,0.0,...,0.0,NaN,NaN,NaN,0.0,0.000,0.0,0.0,0.000000,0.0
3,4,NaN,NaN,NaN,NaN,None,2.00,2.000000,2.00,0.0,...,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,5,NaN,NaN,NaN,NaN,None,0.00,0.000000,0.00,0.0,...,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,1025,13.0,NaN,6.0,NaN,None,1.00,3.785714,40.00,0.0,...,28558.0,NaN,NaN,NaN,0.0,0.000,0.0,0.0,0.000000,0.0
996,1026,9.0,NaN,3.0,NaN,None,0.25,0.583333,1.25,0.0,...,0.0,NaN,NaN,NaN,0.0,0.000,0.0,0.0,0.000000,0.0
997,1027,12.0,NaN,2.0,NaN,None,1.00,3.000000,15.00,0.0,...,32306.0,0.0,5.714286,40.0,0.0,0.000,0.0,0.0,0.000000,0.0
998,1028,NaN,NaN,NaN,NaN,None,1.00,1.800000,3.00,0.0,...,0.0,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.000000,0.0


In [131]:
features = features.merge(
    form_type,
    on=['ein', 'tax_year'],
    how='left'
)

features = features.merge(
    mission,
    on=['ein', 'tax_year'],
    how='left'
)

features = features.merge(
    address,
    on='ein',
    how='left'
)

features = features.merge(
    name,
    on='ein',
    how='left'
)

features = features.merge(
    ot,
    on='filing_id',
    how='left'
)

features = features.merge(
    people,
    on='filing_id',
    how='left'
)

features

,filing_id,ein,tax_year,exempt_organization_type,grants_as_rev_frac,perc_ind_voting_members,political_activity_flag,num_contractors,total_contractor_comp,contractor_compensation_frac,...,max_compensation_org,min_avg_weekly_hours_related_org,avg_avg_weekly_hours_related_org,max_avg_weekly_hours_related_org,min_compensation_related_org,avg_compensation_related_org,max_compensation_related_org,min_compensation_other,avg_compensation_other,max_compensation_other
0,1,272292010,2019,501(c)(3),0.673025,1.0,0.0,1.0,988913.0,0.869091,...,0.0,NaN,NaN,NaN,400.0,16920.625,99665.0,0.0,0.000000,0.0
1,2,911152733,2020,501(c)(2),0.363716,0.0,0.0,NaN,NaN,NaN,...,0.0,NaN,NaN,NaN,0.0,0.000,0.0,0.0,0.000000,0.0
2,3,223519265,2020,501(c)(3),0.072674,1.0,0.0,NaN,NaN,NaN,...,0.0,NaN,NaN,NaN,0.0,0.000,0.0,0.0,0.000000,0.0
3,4,260741074,2020,501(c)(3),NaN,NaN,0.0,NaN,NaN,NaN,...,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,5,814522819,2020,501(c)(3),NaN,NaN,0.0,NaN,NaN,NaN,...,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,96,431879103,2020,501(c)(6),NaN,NaN,0.0,NaN,NaN,NaN,...,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
96,97,320078573,2020,501(c)(3),0.993314,0.0,0.0,NaN,NaN,NaN,...,0.0,NaN,NaN,NaN,0.0,0.000,0.0,0.0,0.000000,0.0
97,98,760190638,2019,501(c)(3),NaN,NaN,0.0,NaN,NaN,NaN,...,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
98,99,223681722,2020,501(c)(3),0.914130,1.0,0.0,NaN,NaN,NaN,...,74885.0,NaN,NaN,NaN,0.0,0.000,0.0,0.0,0.000000,0.0
